# Tutorial 05: Parameter Sensitivity Analysis

Learn how to analyze how different problem parameters affect solution quality and solver behavior.

**What you'll learn:**
- Design sensitivity analysis experiments
- Test impact of time windows, capacity, battery constraints
- Collect and analyze performance metrics
- Make data-driven configuration decisions

**Prerequisites:**
- Tutorial 01 (Quickstart)
- Tutorial 03 (Custom Problems)
- Tutorial 04 (Problem Configurations)

**Time:** ~25 minutes

## 1. Setup and Imports

In [ ]:
# Standard imports
import numpy as np
import pandas as pd
import random
import matplotlib.pyplot as plt

# VRP Toolkit
from vrp_toolkit.problems.pdptw import PDPTWInstance
from vrp_toolkit.algorithms.alns.solver import greedy_insertion_initial_solution

# Set seed for reproducibility
np.random.seed(42)
random.seed(42)

print("Imports successful! Ready for sensitivity analysis.")

## 2. Quick Start: Single Parameter Sweep

Let's start with a simple analysis: **How does vehicle capacity affect solution quality?**

We'll create the same PDPTW instance and solve it with different capacity values.

In [ ]:
# Helper function to create test instance
def create_test_instance():
    """Create a 3-order PDPTW instance for testing."""
    order_table = pd.DataFrame([
        # Depot
        {'ID': 0, 'Type': 'depot', 'X': 0, 'Y': 0, 'Demand': 0,
         'StartTime': 0, 'EndTime': 480, 'ServiceTime': 0, 'PartnerID': 0,
         'RealIndex': 0, 'RealType': 'depot'},
        # Order 1
        {'ID': 1, 'Type': 'cp', 'X': 5, 'Y': 3, 'Demand': 10,
         'StartTime': 0, 'EndTime': 480, 'ServiceTime': 5, 'PartnerID': 4,
         'RealIndex': 1, 'RealType': 'cp'},
        # Order 2
        {'ID': 2, 'Type': 'cp', 'X': 8, 'Y': 2, 'Demand': 8,
         'StartTime': 0, 'EndTime': 480, 'ServiceTime': 5, 'PartnerID': 5,
         'RealIndex': 2, 'RealType': 'cp'},
        # Order 3
        {'ID': 3, 'Type': 'cp', 'X': 3, 'Y': 7, 'Demand': 12,
         'StartTime': 0, 'EndTime': 480, 'ServiceTime': 5, 'PartnerID': 6,
         'RealIndex': 3, 'RealType': 'cp'},
        # Deliveries
        {'ID': 4, 'Type': 'cd', 'X': 12, 'Y': 8, 'Demand': -10,
         'StartTime': 0, 'EndTime': 480, 'ServiceTime': 5, 'PartnerID': 1,
         'RealIndex': 4, 'RealType': 'cd'},
        {'ID': 5, 'Type': 'cd', 'X': 10, 'Y': 12, 'Demand': -8,
         'StartTime': 0, 'EndTime': 480, 'ServiceTime': 5, 'PartnerID': 2,
         'RealIndex': 5, 'RealType': 'cd'},
        {'ID': 6, 'Type': 'cd', 'X': 6, 'Y': 10, 'Demand': -12,
         'StartTime': 0, 'EndTime': 480, 'ServiceTime': 5, 'PartnerID': 3,
         'RealIndex': 6, 'RealType': 'cd'}
    ])
    
    # Compute distance matrix
    n = len(order_table)
    dist_matrix = np.zeros((n, n))
    coords = order_table[['X', 'Y']].values
    
    for i in range(n):
        for j in range(n):
            if i != j:
                dist_matrix[i, j] = np.linalg.norm(coords[i] - coords[j])
    
    time_matrix = dist_matrix / 2.0
    
    return PDPTWInstance(
        order_table=order_table,
        distance_matrix=dist_matrix,
        time_matrix=time_matrix,
        robot_speed=2.0
    )

# Create instance
instance = create_test_instance()

# Test different capacity values
capacities = [15, 20, 25, 30, 40]
results = []

print("Testing different vehicle capacities...")
print(f"Total demand: {instance.order_table[instance.order_table['Demand'] > 0]['Demand'].sum():.0f}\n")

for cap in capacities:
    sol = greedy_insertion_initial_solution(
        problem=instance,
        num_vehicles=3,
        vehicle_capacity=cap,
        battery_capacity=200,
        battery_consume_rate=1,
        penalty_unvisit=1000,
        penalty_delay=50
    )
    
    results.append({
        'capacity': cap,
        'objective': sol.objective_function(),
        'feasible': sol.is_feasible(),
        'routes': len([r for r in sol.routes if len(r) > 2])  # Non-empty routes
    })
    
    print(f"Capacity={cap:2d}: Objective={sol.objective_function():7.2f}, "
          f"Feasible={sol.is_feasible()}, Routes={results[-1]['routes']}")

print("\nKey insight: Larger capacity → Fewer routes needed → Lower objective")

## 3. Multi-Parameter Sensitivity Analysis

### 3.1 Time Window Sensitivity

**Question:** How do tighter time windows affect solution feasibility?

In [ ]:
# Test different time window tightness levels
time_windows = [
    (0, 480),  # Very relaxed (8 hours)
    (0, 240),  # Relaxed (4 hours)
    (0, 120),  # Moderate (2 hours)
    (0, 60),   # Tight (1 hour)
    (0, 30)    # Very tight (30 minutes)
]

tw_results = []

print("Testing time window sensitivity...\n")

for start, end in time_windows:
    # Modify order table
    order_table = instance.order_table.copy()
    order_table['StartTime'] = start
    order_table['EndTime'] = end
    
    # Create new instance
    inst_tw = PDPTWInstance(
        order_table=order_table,
        distance_matrix=instance.distance_matrix,
        time_matrix=instance.time_matrix,
        robot_speed=instance.robot_speed
    )
    
    # Solve
    sol = greedy_insertion_initial_solution(
        problem=inst_tw,
        num_vehicles=3,
        vehicle_capacity=30,
        battery_capacity=200,
        battery_consume_rate=1,
        penalty_unvisit=1000,
        penalty_delay=50
    )
    
    tw_results.append({
        'window_length': end - start,
        'objective': sol.objective_function(),
        'feasible': sol.is_feasible()
    })
    
    print(f"Window [{start:3d}, {end:3d}] ({end-start:3d} min): "
          f"Objective={sol.objective_function():7.2f}, Feasible={sol.is_feasible()}")

print("\nKey insight: Tighter windows may increase objective or become infeasible")

### 3.2 Battery Capacity Sensitivity

In [ ]:
# Test different battery capacities
battery_capacities = [50, 100, 150, 200, 300, 500]
battery_results = []

print("Testing battery capacity sensitivity...\n")

for battery in battery_capacities:
    sol = greedy_insertion_initial_solution(
        problem=instance,
        num_vehicles=3,
        vehicle_capacity=30,
        battery_capacity=battery,
        battery_consume_rate=1,
        penalty_unvisit=1000,
        penalty_delay=50
    )
    
    battery_results.append({
        'battery': battery,
        'objective': sol.objective_function(),
        'feasible': sol.is_feasible()
    })
    
    print(f"Battery={battery:3d}: Objective={sol.objective_function():7.2f}, "
          f"Feasible={sol.is_feasible()}")

print("\nKey insight: Minimum battery threshold needed for feasibility")

## 4. Visualizing Sensitivity Results

Let's create comprehensive visualizations of our sensitivity analysis:

In [ ]:
# Create visualizations
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Capacity sensitivity
cap_df = pd.DataFrame(results)
axes[0, 0].plot(cap_df['capacity'], cap_df['objective'], 'o-', linewidth=2, markersize=8)
axes[0, 0].set_xlabel('Vehicle Capacity')
axes[0, 0].set_ylabel('Objective Value')
axes[0, 0].set_title('Capacity Sensitivity Analysis')
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].axvline(x=30, color='red', linestyle='--', alpha=0.5, label='Total demand')
axes[0, 0].legend()

# Plot 2: Time window sensitivity
tw_df = pd.DataFrame(tw_results)
axes[0, 1].plot(tw_df['window_length'], tw_df['objective'], 's-', 
                linewidth=2, markersize=8, color='green')
axes[0, 1].set_xlabel('Time Window Length (minutes)')
axes[0, 1].set_ylabel('Objective Value')
axes[0, 1].set_title('Time Window Sensitivity Analysis')
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Battery sensitivity
battery_df = pd.DataFrame(battery_results)
axes[1, 0].plot(battery_df['battery'], battery_df['objective'], '^-', 
                linewidth=2, markersize=8, color='orange')
axes[1, 0].set_xlabel('Battery Capacity')
axes[1, 0].set_ylabel('Objective Value')
axes[1, 0].set_title('Battery Capacity Sensitivity Analysis')
axes[1, 0].grid(True, alpha=0.3)

# Plot 4: Feasibility regions
cap_feasible = [r['capacity'] for r in results if r['feasible']]
cap_infeasible = [r['capacity'] for r in results if not r['feasible']]

axes[1, 1].scatter(cap_feasible, [1]*len(cap_feasible), s=100, c='green', 
                   label='Feasible', alpha=0.6, marker='o')
axes[1, 1].scatter(cap_infeasible, [1]*len(cap_infeasible), s=100, c='red', 
                   label='Infeasible', alpha=0.6, marker='x')
axes[1, 1].set_xlabel('Vehicle Capacity')
axes[1, 1].set_yticks([])
axes[1, 1].set_title('Feasibility Regions')
axes[1, 1].legend(loc='upper right')
axes[1, 1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print("Visualizations created successfully!")

## 5. Statistical Analysis

### 5.1 Finding Optimal Parameters

In [ ]:
# Find minimum capacity that keeps solution feasible
feasible_caps = [r['capacity'] for r in results if r['feasible']]
min_feasible_cap = min(feasible_caps) if feasible_caps else None

# Find elbow point (diminishing returns)
cap_df_sorted = cap_df.sort_values('capacity')
obj_improvements = cap_df_sorted['objective'].diff().abs()

print("=== Parameter Recommendations ===")
print(f"\n1. Vehicle Capacity:")
print(f"   Minimum feasible capacity: {min_feasible_cap}")
print(f"   Recommended (balance): ~{min_feasible_cap + 5 if min_feasible_cap else 'N/A'}")

print(f"\n2. Time Window:")
print(f"   All tested windows feasible: {all(r['feasible'] for r in tw_results)}")
shortest_feasible_tw = min([r['window_length'] for r in tw_results if r['feasible']])
print(f"   Shortest feasible window: {shortest_feasible_tw} minutes")

print(f"\n3. Battery Capacity:")
battery_feasible = [r['battery'] for r in battery_results if r['feasible']]
min_battery = min(battery_feasible) if battery_feasible else None
print(f"   Minimum feasible battery: {min_battery}")
print(f"   Recommended (10% buffer): ~{min_battery * 1.1:.0f}" if min_battery else "   N/A")

### 5.2 Parameter Interaction Analysis

Sometimes parameters interact - let's test capacity × time window interaction:

In [ ]:
# Test parameter interactions
capacities_test = [20, 25, 30]
windows_test = [(0, 120), (0, 240), (0, 480)]

interaction_results = []

print("Testing Capacity × Time Window interactions...\n")
print(f"{'Capacity':<12} {'Window':<15} {'Objective':<12} {'Feasible'}")
print("-" * 55)

for cap in capacities_test:
    for start, end in windows_test:
        order_table_int = instance.order_table.copy()
        order_table_int['StartTime'] = start
        order_table_int['EndTime'] = end
        
        inst_int = PDPTWInstance(
            order_table=order_table_int,
            distance_matrix=instance.distance_matrix,
            time_matrix=instance.time_matrix,
            robot_speed=instance.robot_speed
        )
        
        sol = greedy_insertion_initial_solution(
            problem=inst_int,
            num_vehicles=3,
            vehicle_capacity=cap,
            battery_capacity=200,
            battery_consume_rate=1,
            penalty_unvisit=1000,
            penalty_delay=50
        )
        
        interaction_results.append({
            'capacity': cap,
            'window': end - start,
            'objective': sol.objective_function(),
            'feasible': sol.is_feasible()
        })
        
        print(f"{cap:<12} [{start:3d},{end:3d}]     {sol.objective_function():<12.2f} {sol.is_feasible()}")

print("\nKey insight: Some parameter combinations may be infeasible even if each alone is feasible")

## 6. Best Practices for Sensitivity Analysis

**1. Design Systematic Experiments**
- Test one parameter at a time first
- Then test interactions between parameters
- Use consistent baseline for comparisons

**2. Choose Meaningful Ranges**
- Include values below/above expected threshold
- Test extreme cases to find limits
- Use logarithmic spacing for wide ranges

**3. Collect Comprehensive Metrics**
- Objective value (solution quality)
- Feasibility (constraint satisfaction)
- Computation time (efficiency)
- Routes, vehicles used (resource utilization)

**4. Visualize Results**
- Line plots for trends
- Bar charts for comparisons
- Heatmaps for interactions
- Scatter plots for relationships

## 7. Summary

**What you learned:**
- ✅ Design and execute sensitivity analysis experiments
- ✅ Test single parameters (capacity, time windows, battery)
- ✅ Analyze parameter interactions
- ✅ Visualize sensitivity results effectively
- ✅ Make data-driven parameter recommendations

**Key insights from analysis:**
1. **Capacity**: Higher capacity → Lower objective, but diminishing returns
2. **Time Windows**: Tighter windows → Higher objective or infeasibility
3. **Battery**: Minimum threshold needed, extra capacity has little benefit
4. **Interactions**: Parameters can interact in non-obvious ways

**Practical applications:**
- **Fleet sizing**: Find minimum capacity/battery needed
- **Service level design**: Balance time windows with cost
- **Robustness testing**: Ensure solutions work across parameter ranges
- **What-if analysis**: Test scenarios before implementation

**Next steps:**
- Apply these techniques to your own problems
- Test additional parameters (num_vehicles, service times)
- Try Tutorial 06 for custom algorithm sensitivity
- Extend with multi-objective optimization